In [88]:
import pandas as pd

import warnings
warnings.filterwarnings('ignore')

In [89]:
df = pd.read_csv('drone_communication_dataset.csv')


In [90]:
df["label_gps_spoofing"].value_counts()


label_gps_spoofing
0    43434
1     1855
Name: count, dtype: int64

In [91]:
df_filtered = df[(df['label_normal'] == 1) | (df['label_gps_spoofing'] == 1)].copy()

# 3. Zdefiniowanie zmiennej docelowej y
# 1 oznacza atak GPS Spoofing, 0 oznacza ruch normalny (bo odfiltrowaliśmy inne)
y = df_filtered['label_gps_spoofing']

# 4. Czyszczenie zbioru X (Usuwamy wycieki, identyfikatory i szum)
cols_to_drop = [
    # A. Wszystkie labele (bez nich model miałby gotowe odpowiedzi)
    'label_normal', 'label_spoofing', 'label_mitm', 'label_ddos', 
    'label_gps_spoofing', 'label_malware', 'label_jamming', 'label_protocol_exploit',
    
    # B. Wycieki danych (sprzętowe flagi bezpieczeństwa)
    'gps_signal_integrity', 'intrusion_detection_flags', 
    'malware_detection_signals', 'anomaly_in_behavioral_pattern',
    
    # C. Czas i ID (aby uniknąć uczenia się godzin i nazw)
    'timestamp', 'drone_identification',
    
    # D. Tymczasowo usuwamy kolumny o formacie tekstowym/współrzędnych. 
    # Będziesz mógł je dodać później w fazie Feature Engineering po sparsowaniu.
    'drone_gps_coordinates', 'speed_trajectory', 'temporal_patterns'
]

# Bezpieczne usunięcie kolumn (tylko tych, które faktycznie istnieją w dataframe)
cols_to_drop = [c for c in cols_to_drop if c in df_filtered.columns]
X = df_filtered.drop(columns=cols_to_drop)

In [92]:
from sklearn.preprocessing import LabelEncoder
import sklearn as sk

le = LabelEncoder()
y = le.fit_transform(y)

X_train, X_test, y_train, y_test = sk.model_selection.train_test_split(
    X, y, test_size=0.2, shuffle=True, random_state=42
)

In [93]:
X.describe()

,signal_strength,packet_loss_rate,round_trip_time,frequency_band,altitude,transmission_power,message_authentication_status,session_key_validity,signal_noise_ratio,sequence_number_gap,data_rate,network_traffic_volume,uplink_downlink_quality,base_station_load,port_scanning_attempts,drone_signal_handoff
count,6284.000000,6284.000000,6284.000000,6284.000000,6284.000000,6284.000000,6284.000000,6284.000000,6284.000000,6284.000000,6284.000000,6284.000000,6284.000000,6284.000000,6284.000000,6284.0
mean,-60.368555,2.590388,100.533896,183.255983,114.548695,20.795990,0.898472,0.847072,29.704488,0.250796,126.195735,395.275780,51.611871,70.207034,0.092934,1.0
std,5.247138,8.919357,113.785120,362.843320,61.823348,5.034031,0.302050,0.359947,4.200495,1.221117,80.440835,551.360218,9.815409,54.073770,0.762370,0.0
min,-85.000000,0.000000,50.000000,2.400000,21.000000,5.000000,0.000000,0.000000,5.000000,0.000000,50.000000,100.000000,20.000000,50.000000,0.000000,1.0
25%,-60.000000,0.000000,50.000000,2.400000,100.000000,20.000000,1.000000,1.000000,30.000000,0.000000,100.000000,100.000000,50.000000,50.000000,0.000000,1.0
50%,-60.000000,0.000000,50.000000,2.400000,100.000000,20.000000,1.000000,1.000000,30.000000,0.000000,100.000000,100.000000,50.000000,50.000000,0.000000,1.0
75%,-60.000000,0.000000,50.000000,5.000000,100.000000,20.000000,1.000000,1.000000,30.000000,0.000000,100.000000,520.250000,50.000000,50.000000,0.000000,1.0
max,-41.000000,49.000000,499.000000,915.000000,499.000000,49.000000,1.000000,1.000000,49.000000,9.000000,499.000000,1999.000000,99.000000,299.000000,9.000000,1.0


In [94]:
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

num_cols = X.select_dtypes(include=['float64']).columns
cat_cols = X.select_dtypes(include=['object']).columns

preprocessor = ColumnTransformer(
    transformers=[
        ('num', MinMaxScaler(), num_cols),
        ('cat', OneHotEncoder(drop='if_binary', handle_unknown='ignore'), cat_cols)
    ]
)
X_train = preprocessor.fit_transform(X_train)

X_test = preprocessor.transform(X_test)

In [95]:
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

models = {
    'DT' : DecisionTreeClassifier(random_state=42),
    'RF' : RandomForestClassifier(random_state=42),
    'XGBoost' : XGBClassifier(eval_metric='mlogloss',random_state=42),
    'CatBoost' : CatBoostClassifier(verbose=0,random_seed=42),
    'LightGBM' : LGBMClassifier(verbose=-1,random_seed=42),
}

In [96]:
from sklearn.metrics import accuracy_score

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    print(name + ": " + str(accuracy_score(y_test, y_pred)))

DT: 0.6785998408910103
RF: 0.6785998408910103
XGBoost: 0.6785998408910103
CatBoost: 0.6785998408910103
LightGBM: 0.6785998408910103


In [97]:
# Sprawdzenie, co zepsuło wynik (Feature Importance)
import matplotlib.pyplot as plt

xgb_model = models['XGBoost']
importances = pd.Series(xgb_model.feature_importances_, index=preprocessor.get_feature_names_out())

# Wyświetl 10 najbardziej "podejrzanych" kolumn
print(importances.sort_values(ascending=False).head(10))

cat__communication_protocol_ZigBee    0.548133
cat__encryption_type_AES              0.167027
cat__communication_protocol_LoRa      0.129674
cat__encryption_type_Plain-text       0.080443
num__frequency_band                   0.052389
cat__encryption_type_RSA              0.022334
cat__communication_protocol_Wi-Fi     0.000000
dtype: float32
